https://www.kaggle.com/code/aakashnain/tf-jax-tutorials-part1

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

seed=1234
np.random.seed(seed)
tf.random.set_seed(seed)
%config IPCompleter.use_jedi = False

In [2]:
# A zero rank tensor. A zero rank tensor is nothing but a single value
x = tf.constant(5.0)
print(x)

tf.Tensor(5.0, shape=(), dtype=float32)


In [3]:
# We can convert any tensor object to `ndarray` by calling the `numpy()` method
y = tf.constant([1, 2, 3], dtype=tf.int8).numpy()
print(f"`y` is now a {type(y)} object and have a value == {y}")

`y` is now a <class 'numpy.ndarray'> object and have a value == [1 2 3]


In [4]:
# Immutability check

# Rank-1 tensor
x = tf.constant([1, 2], dtype=tf.int8)

# Try to modify the values
try:
    x[1] = 3
except Exception as ex:
    print(type(ex).__name__, ex)

TypeError 'tensorflow.python.framework.ops.EagerTensor' object does not support item assignment


In [5]:
# tf.constant(..) is no special. Let's create a tensor using a diff method
x = tf.ones(2, dtype=tf.int8)
print(x)

try:
    x[0] = 3
except Exception as ex:
    print("\n", type(ex).__name__, ex)

tf.Tensor([1 1], shape=(2,), dtype=int8)

 TypeError 'tensorflow.python.framework.ops.EagerTensor' object does not support item assignment


In [6]:
# Check all the properties of a tensor object
print(f"Shape of x : {x.shape}")
print(f"Another method to obtain the shape using `tf.shape(..)`: {tf.shape(x)}")

print(f"\nRank of the tensor: {x.ndim}")
print(f"dtype of the tensor: {x.dtype}")
print(f"Total size of the tensor: {tf.size(x)}")
print(f"Values of the tensor: {x.numpy()}")

Shape of x : (2,)
Another method to obtain the shape using `tf.shape(..)`: [2]

Rank of the tensor: 1
dtype of the tensor: <dtype: 'int8'>
Total size of the tensor: 2
Values of the tensor: [1 1]


In [7]:
# Create a tensor first. Here is another way
x = tf.cast([1, 2, 3, 4, 5], dtype=tf.float32)
print("Original tensor: ", x)

mask = x%2 == 0
print("Original mask: ", mask)

mask = tf.cast(mask, dtype=x.dtype)
print("Mask casted to original tensor type: ", mask)

# Some kind of operation on an tensor that is of same size 
# or broadcastable to the original tensor. Here we will simply
# use the range object to create that tensor
temp = tf.cast(tf.range(1, 6) * 100, dtype=x.dtype)

# Output tensor
# Input tensor -> [1, 2, 3, 4, 5]
# Mask -> [0, 1, 0, 1, 0]
out = x * (1-mask) + mask * temp
print("Output tensor: ", out)

Original tensor:  tf.Tensor([1. 2. 3. 4. 5.], shape=(5,), dtype=float32)
Original mask:  tf.Tensor([False  True False  True False], shape=(5,), dtype=bool)
Mask casted to original tensor type:  tf.Tensor([0. 1. 0. 1. 0.], shape=(5,), dtype=float32)
Output tensor:  tf.Tensor([  1. 200.   3. 400.   5.], shape=(5,), dtype=float32)


In [8]:
# Another way to achieve the same thing
indices_to_update = tf.where(x % 2 == 0)
print("Indices to update: ", indices_to_update)

# Update the tensor values
updates = [200., 400.]
out = tf.tensor_scatter_nd_update(x, indices_to_update, updates)
print("\nOutput tensor")
print(out)

Indices to update:  tf.Tensor(
[[1]
 [3]], shape=(2, 1), dtype=int64)

Output tensor
tf.Tensor([  1. 200.   3. 400.   5.], shape=(5,), dtype=float32)


In [9]:
# This works!
arr = np.random.randint(5, size=(5,), dtype=np.int32)
print("Numpy array: ", arr)

print("Accessing numpy array elements based on a  condition with irregular strides", arr[[1, 4]])

Numpy array:  [3 4 4 0 1]
Accessing numpy array elements based on a  condition with irregular strides [4 1]


In [11]:
# This doesn't work
try:
    print("Accessing tensor elements based on a  condition with irregular strides", x[[1, 4]])
except Exception as ex:
    print(type(ex).__name__, ex)

InvalidArgumentError {{function_node __wrapped__StridedSlice_device_/job:localhost/replica:0/task:0/device:CPU:0}} Index out of range using input dim 1; input has only 1 dims [Op:StridedSlice] name: strided_slice/


In [12]:
print("Original tensor: ", x.numpy())

# Using the indices that we used for mask
print("\nIndices to update: ", indices_to_update.numpy())

# This works!
print("\n Accesing tensor elements using gather")
print("\n", tf.gather(x, indices_to_update).numpy())

Original tensor:  [1. 2. 3. 4. 5.]

Indices to update:  [[1]
 [3]]

 Accesing tensor elements using gather

 [[2.]
 [4.]]


In [13]:
#  An example with a python list
y = tf.convert_to_tensor([1, 2, 3])
print("Tensor from python list: ", y)

#  An example with a ndarray
y = tf.convert_to_tensor(np.array([1, 2, 3]))
print("Tensor from ndarray: ", y)

#  An example with symbolic tensors
with tf.compat.v1.Graph().as_default():
    y = tf.convert_to_tensor(tf.compat.v1.placeholder(shape=[None, None, None], dtype=tf.int32))
print("Tensor from python list: ", y)

Tensor from python list:  tf.Tensor([1 2 3], shape=(3,), dtype=int32)
Tensor from ndarray:  tf.Tensor([1 2 3], shape=(3,), dtype=int64)
Tensor from python list:  Tensor("Placeholder:0", shape=(None, None, None), dtype=int32)


In [15]:
# String as a tensor object with dtype==tf.string
string = tf.constant("abc", dtype=tf.string)
print("String tensor: ", string)

# String tensors are atomic and non-indexable. 
# This doen't work as expected!
print("\nAccessing second element of the string")
try:
    print(string[1])
except Exception as ex:
    print(type(ex).__name__, ex)

String tensor:  tf.Tensor(b'abc', shape=(), dtype=string)

Accessing second element of the string
InvalidArgumentError {{function_node __wrapped__StridedSlice_device_/job:localhost/replica:0/task:0/device:CPU:0}} Attempting to slice scalar input. [Op:StridedSlice] name: strided_slice/


In [16]:
# This works!
y = [[1, 2, 3],
     [4, 5],
     [6]
    ]

ragged = tf.ragged.constant(y)
print("Creating ragged tensor from python sequence: ", ragged)

Creating ragged tensor from python sequence:  <tf.RaggedTensor [[1, 2, 3], [4, 5], [6]]>


In [17]:
# This won't work
print("Trying to create tensor from above python sequence\n")
try:
    z = tf.constant(y)
except Exception as ex:
    print(type(ex).__name__, ex)

Trying to create tensor from above python sequence

ValueError Can't convert non-rectangular Python sequence to Tensor.


In [18]:
# Let's say you have a an array like this one
# [[1 0 0]
#  [0 2 0]
#  [0 0 3]]
# If there are too many zeros in your `huge` tensor, then it is wise to use `sparse`
# tensors instead of `dense` one. Let's say how to create this one. We need to specify:
# 1. Indices where our values are
# 2. The values 
# 3. The actual shape

sparse_tensor = tf.SparseTensor(indices=[[0, 0], [1, 1], [2, 2]],
                                values=[1, 2, 3],
                                dense_shape=[3, 3]
                               )
print(sparse_tensor)

# You can convert sparse tensors to dense as well
print("\n", tf.sparse.to_dense(sparse_tensor))

SparseTensor(indices=tf.Tensor(
[[0 0]
 [1 1]
 [2 2]], shape=(3, 2), dtype=int64), values=tf.Tensor([1 2 3], shape=(3,), dtype=int32), dense_shape=tf.Tensor([3 3], shape=(2,), dtype=int64))

 tf.Tensor(
[[1 0 0]
 [0 2 0]
 [0 0 3]], shape=(3, 3), dtype=int32)
